## FFT filter

Now, we will try to use FFT to denoise the FIB-SEM image. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from imageio import imread

In [ ]:
img = imread("AIST_LSCF_300h_055.tif")

print(img.shape)

img = img.astype(float)

plt.figure(figsize=(10,5))
plt.imshow(img, cmap='gray')
plt.axis('off')
plt.show()

I_norm = img/255

print(np.max(img), np.min(img))

### Vertical streaks

You can observe that there are some vertical streaks resulting from FIB milling. Let's FFT this image and check its Fourier spectral.

In [ ]:
Fs = np.fft.fft2(I_norm)

Amp = np.abs(Fs)

lgA = np.log10(Amp)

plt.subplot(2,1,1)
plt.imshow(Amp)

plt.subplot(2,1,2)
plt.imshow(lgA)

plt.show()

FFT shift the spectral.



In [ ]:
F_shift = np.fft.fftshift(lgA)
plt.imshow(F_shift)
plt.show()

We can see that there are some stripes in  the Fourier space.If we remove those stripes, the vertical periodicity in the FIB-SEM image may be removed. 

- Let's try whether low-pass filter can work.

In [ ]:
# -----------------------------
# 3. Low-pass filter
# -----------------------------
rows, cols = I_norm.shape
crow, ccol = rows // 2, cols // 2

R = 30   # cutoff radius; tune this

y, x = np.ogrid[:rows, :cols]
distance = np.sqrt((x - ccol)**2 + (y - crow)**2)

mask = distance <= R

F_filtered = F_shift * mask


plt.figure(figsize=(6, 6))

plt.imshow(np.log(1+ np.abs(F_filtered)), cmap="gray")
plt.title("filtered FFT spectrum")
plt.colorbar()
plt.axis("off")

In [ ]:
# -----------------------------
# 4. Inverse FFT
# -----------------------------
F_ishift = np.fft.ifftshift(F_filtered)
I_denoised = np.fft.ifft2(F_ishift)
I_denoised = np.abs(I_denoised)


In [ ]:
# -----------------------------
# 5. Plot results
# -----------------------------
plt.figure(figsize=(14, 5))

plt.subplot(1, 3, 1)
plt.imshow(I_norm, cmap="gray")
plt.title("Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(np.log(1 + np.abs(F_shift)), cmap="gray")
plt.title("FFT Spectrum")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(I_denoised, cmap="gray")
plt.title("FFT Low-pass Denoised")
plt.axis("off")

plt.show()

### Remove stripes in Fourier space.
Now, let's create a mask to remove only the vertical or horizontal stripes.

In [ ]:
F = np.fft.fft2(I_norm)
Fshift = np.fft.fftshift(F)

ny, nx = img.shape
cy, cx = ny//2, nx//2

Y, X = np.ogrid[:ny, :nx]

mask = np.ones((ny, nx))

# remove horizontal band
band_width = 4
row_mask = np.abs(np.arange(ny) - cy) < band_width
mask[row_mask, :] = 0

# band_width = 2
# col_mask = np.abs(np.arange(nx) - cx) < band_width
# mask[:, col_mask] = 0

# preserve central low frequencies
r_keep = 10

center = (X - cx)**2 + (Y - cy)**2 < r_keep**2

mask[center] = 1

F_filt = Fshift * mask

img_clean = np.real(np.fft.ifft2(np.fft.ifftshift(F_filt)))

plt.figure(figsize=(5,12))

plt.subplot(311)
plt.imshow(I_norm, cmap='gray')
plt.title("Original")

plt.subplot(312)
plt.imshow(np.log1p(np.abs(Fshift)), cmap='viridis')
plt.title("FFT")

plt.subplot(313)
plt.imshow(np.log1p(np.abs(F_filt)), cmap='viridis')
plt.title("FFT")


plt.show()

In [ ]:
plt.imshow(img_clean, cmap='gray')
plt.title("Filtered")

plt.show()